# ARGO NLP Routine - Bedrock integration

### Author: Rob Methven (methvenr@amazon.com)
### Date: 7/18/2024

In [30]:
import json
import boto3
import pandas as pd
import os
from botocore.config import Config
import numpy as np
import re

# Delete Old Data

In [31]:
# Delete the latest extract file from S3
s3 = boto3.client('s3')

s3.delete_object(Bucket = 'wwetc-data-extracts', Key = 'wwfe-NLP-analysis/Loads/NLP_Verbatim_Processed_sub.csv')

{'ResponseMetadata': {'RequestId': '7V0JC3ZBC58KJJW2',
  'HostId': 'pwz9eXnuNFD/BcfYvTVdO+MHjlJZt8SxZ5u8onBdjpE84n0pJmjc958VzPCEH5D7hYX5dqrk7ibfYqPIcNGiyg==',
  'HTTPStatusCode': 204,
  'HTTPHeaders': {'x-amz-id-2': 'pwz9eXnuNFD/BcfYvTVdO+MHjlJZt8SxZ5u8onBdjpE84n0pJmjc958VzPCEH5D7hYX5dqrk7ibfYqPIcNGiyg==',
   'x-amz-request-id': '7V0JC3ZBC58KJJW2',
   'date': 'Fri, 04 Oct 2024 16:19:15 GMT',
   'x-amz-version-id': 'Cw5UsKVy2342zfyp5PhXuPOUvD5DHjnX',
   'x-amz-delete-marker': 'true',
   'server': 'AmazonS3'},
  'RetryAttempts': 1},
 'DeleteMarker': True,
 'VersionId': 'Cw5UsKVy2342zfyp5PhXuPOUvD5DHjnX'}

# Import data

In [32]:
# import extract from reshift/S3 to be processed
s3 = boto3.client('s3')
obj = s3.get_object(Bucket = 'wwetc-data-extracts',Key = 'wwfe-NLP-analysis/Extracts/NLP_new000.csv')

df = pd.read_csv(obj['Body'], delimiter = ',')

In [33]:
# reimport original file to label unprocessed data for export
obj = s3.get_object(Bucket = 'wwetc-data-extracts',Key = 'wwfe-NLP-analysis/Extracts/NLP_new000.csv')
df_unproc = pd.read_csv(obj['Body'], delimiter = ',')

#drop all columns but idkey from unprocessed ref list
df_unproc = df_unproc['idkey']

# remove duplicates
df_unproc = df_unproc.drop_duplicates()

# Text Processing

In [34]:
# limit string length to maximum allowable length (10000)
df['verbatim'] = df['verbatim'].str[:10000]

# drop na's
df = df.dropna()

# remove pipe character
df['verbatim'] = df['verbatim'].str.replace('|', '', regex = True)

# drop duplicates
df = df.drop_duplicates()

# reset index
df = df.reset_index(drop = True)

# Translate non-English text

In [35]:
# initialize Translate module
translate = boto3.client(service_name = 'translate', region_name = "us-east-2", config=Config(connect_timeout=5, read_timeout=60, retries={'max_attempts': 20}))

#initialize Comprehend module
comprehend = boto3.client('comprehend', config=Config(connect_timeout=5, read_timeout=60, retries={'max_attempts': 20}))

In [36]:
# Isolate verbatim field
language_input = list(df.verbatim)

# Create target output list for translation results
language_output = []

# Detect dominant language
for y in language_input:
    language_output.append(comprehend.detect_dominant_language(Text = y))

In [37]:
# Create df from json list
language_result = pd.json_normalize(language_output)

# extract dominant language code
language_translation = language_result['Languages'].apply(lambda x: x[0]['LanguageCode'])

# extract dominant language code confidence score
language_conf = language_result['Languages'].apply(lambda x: x[0]['Score'])

# append langauge metrics to df
df = df.assign(language_original = language_translation, language_score = language_conf)

# drop df rows with unsupported languages
sup_lang = ['zh','zh-TW','en','fr','fr-CA','ja','ko','es','es-MX']

df = df[df.language_original.isin(sup_lang) == True]

# reset index
df = df.reset_index(drop = True)

# Drop all rows that have a language score less than 45%
df = df.loc[df['language_score'] > 0.45]

In [39]:
# Filter non-english verbatim to be translated
noneng = df[df['language_original'] != 'en']

# filter English verbatim to be joined to translated text
eng = df[df['language_original'] == 'en']

# Drop unrequired columns
eng = eng.drop(columns=['language_original'])
eng = eng.drop(columns=['language_score'])

# add required columns
eng['TranslatedText'] = eng['verbatim'].copy()
eng['SourceLanguageCode'] = 'en'
eng['TargetLanguageCode'] = 'en'

In [40]:
# Isolate verbatim field
translate_input = list(noneng.verbatim)

# Create target output list for translation results
translate_output = []

# run batch translation over all verbatim entries
for x in translate_input:
    translate_output.append(translate.translate_text(Text=x, SourceLanguageCode="auto", TargetLanguageCode="en"))

In [41]:
if len(translate_output) > 0:

    # Convert json list to df
    translate_result = pd.json_normalize(translate_output)

    # Drop columns that are not required
    translate_result = translate_result[['TranslatedText','SourceLanguageCode','TargetLanguageCode']]

    # Reset df index
    noneng = noneng.reset_index(drop = True)

    # join translation results to original df
    noneng = noneng.join(translate_result)

    # Drop unrequired columns from processing routine
    noneng = noneng.drop(columns=['language_original'])
    noneng = noneng.drop(columns=['language_score'])
    
    # concatenating english and non-english
    df = pd.concat([eng, noneng], axis=0).reset_index(drop = True)
    
else:
    
    df = eng

In [42]:
# Create translated verbatim column
df['verbatim_translated'] = df.loc[:,'TranslatedText']

In [43]:
# Reorder Columns
df.insert(len(df.columns)-1, 'TranslatedText', df.pop('TranslatedText'))
df.insert(len(df.columns)-1, 'SourceLanguageCode', df.pop('SourceLanguageCode'))
df.insert(len(df.columns)-1, 'TargetLanguageCode', df.pop('TargetLanguageCode'))

In [44]:
# Rename fields
df = df.rename(columns={"verbatim": "verbatim_original", "TranslatedText": "verbatim", "SourceLanguageCode": "language_original", "TargetLanguageCode": "language_translation"})

# Text processing

In [45]:
# convert to lower case
df['verbatim'] = df['verbatim'].str.lower()

# Strip white spaces
df['verbatim'] = df['verbatim'].str.strip()

# remove punctuation
import string
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df["verbatim"] = df["verbatim"].apply(remove_punctuation)
#df["verbatim_original"] = df["verbatim_original"].apply(remove_punctuation)

# Replace erroneous words that comprehend IDs as 'neagative' with blank space
df['verbatim'] = df['verbatim'].str.replace('nothing', '')
df['verbatim'] = df['verbatim'].str.replace('none', '')
df['verbatim'] = df['verbatim'].str.replace('not', '')
df['verbatim'] = df['verbatim'].str.replace('no feedback', '')
df['verbatim'] = df['verbatim'].str.replace('no suggestions', '')
df['verbatim'] = df['verbatim'].str.replace('no suggestion', '')
df['verbatim'] = df['verbatim'].str.replace('no comments', '')
df['verbatim'] = df['verbatim'].str.replace('no comment', '')
df['verbatim'] = df['verbatim'].str.replace('no ', '')

#remove useless comments
remove_list = ['none','nothing','na','nope','none.','n.a.','n/a']

df = df[~df['verbatim'].isin(remove_list)]

# remove comments with less than 3 characters
df = df.loc[df['verbatim'].str.len() > 3]

# reset index
df = df.reset_index(drop = True)

# Sentiment Analysis

In [46]:
# Truncate verbatim strings to maximum allowable length for analysis
df['verbatim'] = df['verbatim'].str[:5000]

# Create a batch
verbatim_list = list(df.verbatim)

# Create target output list for comprehend results
verbatim_output = []

#Run a sentiment batch
for v in verbatim_list:
    verbatim_output.append(comprehend.detect_sentiment(Text=v,LanguageCode='en'))

In [47]:
# Convert json list to df
verbatim_result = pd.json_normalize(verbatim_output)

# Drop columns that are not required
verbatim_result = verbatim_result[['SentimentScore.Positive','SentimentScore.Negative','SentimentScore.Neutral','SentimentScore.Mixed','Sentiment']]

# Rename fields
verbatim_result = verbatim_result.rename(columns={"SentimentScore.Positive": "Positive_Score", "SentimentScore.Negative": "Negative_Score",
                                                  "SentimentScore.Neutral": "Neutral_Score", "SentimentScore.Mixed": "Mixed_Score"})

# Reset df index
verbatim_result = verbatim_result.reset_index(drop = True)

In [48]:
# Join dfs
output = df.join(verbatim_result)

In [49]:
# Round floats in df to 3 decimals (required for importing to Redshift)
final_result = output.round(3)

In [50]:
# merge unprocessed data
export = df_unproc.to_frame().merge(final_result, how = 'left', on = 'idkey')

# check non-processed rows
#export[export['Sentiment'].isna()]

# Clean up delimiters and extra characters for clean export
export['verbatim'] = export['verbatim'].str.replace('"','', regex = True)
export['verbatim'] = export['verbatim'].str.replace('|','', regex = True)
export['verbatim_original'] = export['verbatim_original'].str.replace('"','', regex = True)
export['verbatim_original'] = export['verbatim_original'].str.replace('|','', regex = True)

# Top Level Topic Modelling - Bedrock

In [51]:
# Call the bedrock service using the API
bedrock = boto3.client(service_name='bedrock-runtime', region_name = "us-west-2", config=Config(connect_timeout=5, read_timeout=90, retries={'max_attempts': 20}))

# Call S3 service
s3 = boto3.client('s3')

In [52]:
topic_model_input = export.dropna().reset_index(drop = True)

In [53]:
gen = []

for index, row in topic_model_input.iterrows():

    text = row['verbatim_translated']
    
    # Set up prompt - add as much context and instruction as required. Specifics are critical to generating good responses

    prompt = """The following text is general feedback on training content. Given this context, define a single, most appropriate topic.
                Restrict the topic to one of either 'Training Length', 'Technical Issues', 'Training Relevance', 'Design, Format and Structure'. Do not provide additional topic designations.
                If you are not confident that the text falls into one of the predefined categories, label it as 'General Feedback'.
                The Response should only include the topic name and nothing else."""

    body = json.dumps({
            "max_tokens": 4096,
            "messages": [{"role": "user", "content": prompt + text}],
            "anthropic_version": "bedrock-2023-05-31"
    }) 

 
    response = bedrock.invoke_model(body = body, modelId = "anthropic.claude-3-sonnet-20240229-v1:0")

    response_body_name = json.loads(response.get("body").read())
    
    response_body_name = response_body_name.get("content")

    response_body_df = pd.DataFrame(response_body_name)

    gen.append(response_body_df.loc[0, 'text'])

In [54]:
# Covert list to dataframe
gen_df = pd.DataFrame(gen)

# Rename output column
gen_df = gen_df.rename(columns={0: "topic"})

# Strip out any white space
gen_df['topic'] = gen_df['topic'].str.strip()

# Specify labels expected in output
gen_labels = ['Training Length', 'Technical Issues', 'Training Relevance', 'Design, Format and Structure','General Feedback']

# Bucket any outputs that are generated that are not the specified labels as 'Other'
gen_df['topic'] = np.where(~np.isin(gen_df['topic'], gen_labels), 'General Feedback', gen_df['topic'])

# Merge output with original dataset
gen_df = topic_model_input.merge(gen_df['topic'], left_index=True, right_index=True)

# Sub topic modelling - Bedrock

In [55]:
# import data
sub_topic_input = gen_df[["idkey", "verbatim_translated","topic"]]

## Training Length

In [56]:
# Extract Training Length data
training_length = sub_topic_input[sub_topic_input['topic'] == 'Training Length'].reset_index(drop = True)

In [ ]:
import json
import pandas as pd
import numpy as np

def classify_training_feedback(training_length):
    if len(training_length) > 0:
        length = []
        for index, row in training_length.iterrows():
            text = row['verbatim_translated']
            prompt = """The following text is feedback on training videos. Given this context, define a single, most appropriate topic. Topics should be restricted to one of either 'Too Long', 'Just Right', 'Too Short', 'Time Estimate'.
                        For context, feedback like shorter should be classified as 'Too Long' and feedback like 'too much in such a short time' should be classified as 'Too Short' .
                        The Response should only include the topic name and nothing else."""
            body = json.dumps({
                "max_tokens": 4096,
                "messages": [{"role": "user", "content": prompt + text}],
                "anthropic_version": "bedrock-2023-05-31"
            })
            response = bedrock.invoke_model(body=body, modelId="anthropic.claude-3-sonnet-20240229-v1:0")
            response_body_name = json.loads(response.get("body").read())
            response_body_name = response_body_name.get("content")
            response_body_df = pd.DataFrame(response_body_name)
            length.append(response_body_df.loc[0, 'text'])
        
        length_df = pd.DataFrame(length)
        length_df = length_df.rename(columns={0: "sub_topic"})
        length_df['sub_topic'] = length_df['sub_topic'].str.strip()
        length_labels = ['Too Long', 'Just Right', 'Too Short', 'Time Estimate']
        length_df['sub_topic'] = np.where(~np.isin(length_df['sub_topic'], length_labels), 'Other', length_df['sub_topic'])
        length_df = training_length.merge(length_df['sub_topic'], left_index=True, right_index=True)
    else:
        col_names = ['idkey', 'verbatim_translated', 'topic', 'sub_topic']
        length_df = pd.DataFrame(columns=col_names)
    
    return length_df

In [57]:
if len(training_length) > 0:
    
    length = []

    for index, row in training_length.iterrows():

        text = row['verbatim_translated']    
        # Set up prompt - add as much context and instruction as required. Specifics are critical to generating good responses

        prompt = """The following text is feedback on training videos. Given this context, define a single, most appropriate topic. Topics should be restricted to one of either 'Too Long', 'Just Right', 'Too Short', 'Time Estimate'.
                    For context, feedback like shorter should be classified as 'Too Long' and feedback like 'too much in such a short time' should be classified as 'Too Short' .
                    The Response should only include the topic name and nothing else."""

        body = json.dumps({
                "max_tokens": 4096,
                "messages": [{"role": "user", "content": prompt + text}],
                "anthropic_version": "bedrock-2023-05-31"
        })

 
        response = bedrock.invoke_model(body = body, modelId = "anthropic.claude-3-sonnet-20240229-v1:0")
        response_body_name = json.loads(response.get("body").read())
        response_body_name = response_body_name.get("content")
        response_body_df = pd.DataFrame(response_body_name)
        length.append(response_body_df.loc[0, 'text'])
    
    # Covert list to dataframe
    length_df = pd.DataFrame(length)
    # Rename output column
    length_df = length_df.rename(columns={0: "sub_topic"})
    # Strip out any white space
    length_df['sub_topic'] = length_df['sub_topic'].str.strip()
    # Specify labels expected in output
    length_labels = ['Too Long', 'Just Right','Too Short','Time Estimate']
    # Bucket any outputs that are generated that are not the specified labels as 'Other'
    length_df['sub_topic'] = np.where(~np.isin(length_df['sub_topic'],length_labels), 'Other', length_df['sub_topic'])
    # Merge output with original dataset
    length_df = training_length.merge(length_df['sub_topic'], left_index=True, right_index=True)
    
else:

    # column name list  
    col_names =  ['idkey', 'verbatim_translated','topic','sub_topic']
    # create an empty dataframe 
    length_df  = pd.DataFrame(columns = col_names) 

## Techincal Issues

In [58]:
# Extract Technical Issues data
tech = sub_topic_input[sub_topic_input['topic'] == 'Technical Issues'].reset_index(drop = True)

In [59]:
if len(tech) > 0:

    technical = []
    
    for index, row in tech.iterrows():

        text = row['verbatim_translated']
    
        # Set up prompt - add as much context and instruction as required. Specifics are critical to generating good responses

        prompt = """The following text is feedback on technical issues with training videos. Given this context, define a single, most appropriate topic.
                    Topics should be restricted to one of either 'Access', 'Controls', 'navigation', 'audio/video issues', 'completion tracking issues', 'inability to change answers', 'vpn requirement'
                    If you are not confident that the text falls into one of the predefined categories, label it as 'Other'.
                    The Response should only include the topic name and nothing else."""

        body = json.dumps({
                "max_tokens": 4096,
                "messages": [{"role": "user", "content": prompt + text}],
                "anthropic_version": "bedrock-2023-05-31"
        })

 
        response = bedrock.invoke_model(body = body, modelId = "anthropic.claude-3-sonnet-20240229-v1:0")

        response_body_name = json.loads(response.get("body").read())
        
        response_body_name = response_body_name.get("content")

        response_body_df = pd.DataFrame(response_body_name)

        technical.append(response_body_df.loc[0, 'text'])
        
    # Covert list to dataframe
    tech_df = pd.DataFrame(technical)

    # Rename output column
    tech_df = tech_df.rename(columns={0: "sub_topic"})

    # Strip out any white space
    tech_df['sub_topic'] = tech_df['sub_topic'].str.strip()

    # Specify labels expected in output
    tech_labels = ['Access', 'Controls', 'navigation', 'audio/video issues', 'completion tracking issues', 'inability to change answers', 'vpn requirement']

    # Bucket any outputs that are generated that are not the specified labels as 'Other'
    tech_df['sub_topic'] = np.where(~np.isin(tech_df['sub_topic'],tech_labels), 'Other', tech_df['sub_topic'])

    # Merge output with original dataset
    tech_df = tech.merge(tech_df['sub_topic'], left_index=True, right_index=True)
    
else:
        
    # column name list  
    col_names =  ['idkey', 'verbatim_translated','topic','sub_topic']
  
    # create an empty dataframe 
    tech_df  = pd.DataFrame(columns = col_names) 

## Relevance

In [60]:
# Extract Relevance data
relevance = sub_topic_input[sub_topic_input['topic'] == 'Training Relevance'].reset_index(drop = True)

In [61]:
if len(relevance) > 0:
    
    rel = []

    for index, row in relevance.iterrows():

        text = row['verbatim_translated']
    
        # Set up prompt - add as much context and instruction as required. Specifics are critical to generating good responses

        prompt = """The following text is feedback on the relevance of training content. Given this context, define a single, most appropriate topic.
                    Restrict the topic to one of either 'Informative', 'Detail and Depth', 'Examples and Case Studies', 'Incorrect Information'. Do not provide additional topic designations.
                    If you are not confident that the text falls into one of the predefined categories, label it as 'Other'.
                    The Response should only include the topic name and nothing else."""

        body = json.dumps({
            "max_tokens": 4096,
            "messages": [{"role": "user", "content": prompt + text}],
            "anthropic_version": "bedrock-2023-05-31"
        })

 
        response = bedrock.invoke_model(body = body, modelId = "anthropic.claude-3-sonnet-20240229-v1:0")

        response_body_name = json.loads(response.get("body").read())
    
        response_body_name = response_body_name.get("content")

        response_body_df = pd.DataFrame(response_body_name)

        rel.append(response_body_df.loc[0, 'text'])
    
    # Covert list to dataframe
    rel_df = pd.DataFrame(rel)

    # Rename output column
    rel_df = rel_df.rename(columns={0: "sub_topic"})

    # Strip out any white space
    rel_df['sub_topic'] = rel_df['sub_topic'].str.strip()

    # Specify labels expected in output
    rel_labels = ['Informative', 'Detail and Depth', 'Examples and Case Studies', 'Incorrect Information']

    # Merge output with original dataset
    rel_df['sub_topic'] = np.where(~np.isin(rel_df['sub_topic'],rel_labels), 'Other', rel_df['sub_topic'])

    # Bucket any outputs that are generated that are not the specified labels as 'Other'
    rel_df = relevance.merge(rel_df['sub_topic'], left_index=True, right_index=True)
       
else:
        
    # column name list  
    col_names =  ['idkey', 'verbatim_translated','topic','sub_topic']
  
    # create an empty dataframe 
    rel_df  = pd.DataFrame(columns = col_names) 

## Design, Format and Structure

In [62]:
# Extract Design, Format and Structure data
dfs = sub_topic_input[sub_topic_input['topic'] == 'Design, Format and Structure'].reset_index(drop = True)

In [63]:
if len(dfs) > 0:
    
    design = []

    for index, row in dfs.iterrows():

        text = row['verbatim_translated']
    
        # Set up prompt - add as much context and instruction as required. Specifics are critical to generating good responses

        prompt = """The following text is feedback on the relevance of training content. Given this context, define a single, most appropriate topic.
                    Restrict the topic to one of either 'Visual aids or multimedia', 'Content value', 'Formatting and layout', 'Interactivity and engagement' or 'Structure and flow'. Do not provide additional topic designations.
                    If you are not confident that the text falls into one of the predefined categories, label it as 'Other'.
                    The Response should only include the topic name and nothing else."""

        body = json.dumps({
            "max_tokens": 4096,
            "messages": [{"role": "user", "content": prompt + text}],
            "anthropic_version": "bedrock-2023-05-31"
        }) 

 
        response = bedrock.invoke_model(body = body, modelId = "anthropic.claude-3-sonnet-20240229-v1:0")

        response_body_name = json.loads(response.get("body").read())
    
        response_body_name = response_body_name.get("content")

        response_body_df = pd.DataFrame(response_body_name)

        design.append(response_body_df.loc[0, 'text'])
        
    # Covert list to dataframe
    dfs_df = pd.DataFrame(design)

    # Rename output column
    dfs_df = dfs_df.rename(columns={0: "sub_topic"})

    # Strip out any white space
    dfs_df['sub_topic'] = dfs_df['sub_topic'].str.strip()

    # Specify labels expected in output
    dfs_labels = ['Visual aids or multimedia', 'Content value', 'Formatting and layout', 'Interactivity and engagement', 'Structure and flow', 'Other']

    # Bucket any outputs that are generated that are not the specified labels as 'Other'
    dfs_df['sub_topic'] = np.where(~np.isin(dfs_df['sub_topic'],dfs_labels), 'Other', dfs_df['sub_topic'])

    # Merge output with original dataset
    dfs_df = dfs.merge(dfs_df['sub_topic'], left_index=True, right_index=True)
     
else:
        
    # column name list  
    col_names =  ['idkey', 'verbatim_translated','topic','sub_topic']
  
    # create an empty dataframe 
    dfs_df  = pd.DataFrame(columns = col_names)

## Consolidate dataframes for output

In [64]:
# Create df for General Feedback
gen_feed_df = sub_topic_input[sub_topic_input['topic'] == 'General Feedback'].reset_index(drop = True)

# List the dataframes to combine
frames = [length_df, tech_df, rel_df, dfs_df,gen_feed_df]
# frames = [x for x in frames if len(x) <> 0]

# Concatenate the dataframes
output = pd.concat(frames).reset_index(drop = True)

# Drop duplicate verbatim_translated column
output = output.drop(columns=['verbatim_translated'])

# Merge dataframes for final output to S3
final_output = export.merge(output, on = 'idkey', how = 'left')

# Merge dataframes for final output legacy version to S3
final_output_old = final_output.loc[:, final_output.columns != 'sub_topic']

# Drop duplicates
final_output = final_output.drop_duplicates()
final_output_old = final_output_old.drop_duplicates()

# Export Data

In [65]:
# Export CSV to S3
file_name = "NLP_Verbatim_Processed_sub.csv"

final_output.to_csv(file_name, index = False, sep = '|')

s3.upload_file(file_name, 'wwetc-data-extracts', 'wwfe-NLP-analysis/Loads/NLP_Verbatim_Processed_sub.csv')

# Data clean up

In [66]:
# Delete the latest extract file from S3
s3 = boto3.client('s3')

s3.delete_object(Bucket = 'wwetc-data-extracts', Key = 'wwfe-NLP-analysis/Extracts/NLP_new000.csv')

{'ResponseMetadata': {'RequestId': '3PKVS9G32MZHYA4X',
  'HostId': '2jR66BYdZUwrHE52AFNBTp+yjeUhsWZ7uvJdulVBuQ8W8naGA/nwHa0SeHveQHcadmG4DZIhyEddGDfK07Zv4w==',
  'HTTPStatusCode': 204,
  'HTTPHeaders': {'x-amz-id-2': '2jR66BYdZUwrHE52AFNBTp+yjeUhsWZ7uvJdulVBuQ8W8naGA/nwHa0SeHveQHcadmG4DZIhyEddGDfK07Zv4w==',
   'x-amz-request-id': '3PKVS9G32MZHYA4X',
   'date': 'Fri, 04 Oct 2024 16:20:18 GMT',
   'x-amz-version-id': 'fV56TPLPvS.Rj_S7ajFQo0sYnVJ5K_rZ',
   'x-amz-delete-marker': 'true',
   'server': 'AmazonS3'},
  'RetryAttempts': 1},
 'DeleteMarker': True,
 'VersionId': 'fV56TPLPvS.Rj_S7ajFQo0sYnVJ5K_rZ'}